# Home Credit Default Risk — application-only end-to-end

Mục tiêu của notebook là tạo một baseline có thể kiểm chứng trên bảng application. Feature engineering nằm ở notebook để phục vụ học tập/khám phá; loader, model runner, CV, tuning và submission được import từ `src/credit_scoring`.

## Quy ước experiment

- `TARGET=1` là application có khó khăn trả nợ.
- Primary metric là ROC-AUC.
- Validation mặc định là StratifiedKFold 5 folds, seed 42.
- Không dùng SMOTE, accuracy hoặc leaderboard để tune.
- Chỉ xử lý application ở lần chạy này; các bảng 1-n sẽ được thêm sau khi baseline ổn.

In [ ]:
# 1. Clone source code và import dependency có sẵn trên Kaggle
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

REPO_URL = "https://github.com/ManhTanTran/Qaci-datascience.git"
REPO_BRANCH = "main"
REPO_COMMIT = None  # Optional full commit SHA for reproducible reruns.
REPO_DIR = Path("/kaggle/working/Qaci-datascience")

def run_git(*arguments: str) -> None:
    """Run one git command and fail loudly in the Kaggle notebook."""
    subprocess.run(["git", "-C", str(REPO_DIR), *arguments], check=True)

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
elif REPO_COMMIT is None:
    run_git("checkout", REPO_BRANCH)
    run_git("pull", "--ff-only", "origin", REPO_BRANCH)

if REPO_COMMIT is not None:
    run_git("fetch", "--depth", "1", "origin", REPO_COMMIT)
    run_git("checkout", "--detach", REPO_COMMIT)

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print(f"Repository: {REPO_DIR}")
print(f"Git commit: {GIT_COMMIT}")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from credit_scoring.artifacts import export_dataframe_artifact, export_json_artifact
from credit_scoring.data.home_credit import (
    find_home_credit_data_dir,
    load_home_credit_data,
    summarize_loaded_tables,
)
from credit_scoring.modeling.lightgbm_model import run_lightgbm_cv
from credit_scoring.reproducibility import set_global_seed
from credit_scoring.submission.home_credit import create_home_credit_submission

set_global_seed(42)
pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 160)

In [ ]:
# 2. Tập trung toàn bộ configuration ở một cell
RUN_MODES = {
    "smoke": {
        "sample_size": 5_000,
        "n_splits": 3,
        "n_estimators": 300,
        "early_stopping_rounds": 50,
        "tuning": False,
        "submission": False,
    },
    "baseline": {
        "sample_size": None,
        "n_splits": 5,
        "n_estimators": 5_000,
        "early_stopping_rounds": 200,
        "tuning": False,
        "submission": True,
    },
}
CONFIG = {
    "experiment_name": "E01_application_baseline",
    "run_mode": "smoke",
    "data_dir": None,
    "output_dir": "/kaggle/working/home_credit_outputs",
    "shuffle": True,
    "random_state": 42,
    "n_trials": 30,
    "run_ablation": False,
}
if CONFIG["run_mode"] not in RUN_MODES:
    raise ValueError(f"Unknown run_mode: {CONFIG['run_mode']}")
MODE_CONFIG = RUN_MODES[CONFIG["run_mode"]]
MODEL_CONFIG = {
    "learning_rate": 0.02,
    "n_estimators": MODE_CONFIG["n_estimators"],
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 80,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "random_state": CONFIG["random_state"],
    "n_jobs": -1,
    "verbosity": -1,
}
VALIDATION_CONFIG = {
    "n_splits": MODE_CONFIG["n_splits"],
    "shuffle": CONFIG["shuffle"],
    "random_state": CONFIG["random_state"],
    "early_stopping_rounds": MODE_CONFIG["early_stopping_rounds"],
    "keep_models": True,
}
OUTPUT_DIR = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(CONFIG)

In [ ]:
# 3. Load application tables qua repository loader
DATA_DIR = find_home_credit_data_dir(CONFIG["data_dir"])
print(f"Data directory: {DATA_DIR}")
data = load_home_credit_data(
    data_dir=DATA_DIR,
    tables=["application_train", "application_test"],
    nrows=MODE_CONFIG["sample_size"],
    reduce_memory=True,
    validate=True,
)
train_raw = data["application_train"].copy()
test_raw = data["application_test"].copy()
display(summarize_loaded_tables(data))
display(train_raw["TARGET"].value_counts(normalize=True).rename("target_rate"))

In [ ]:
# 4. Data audit: missingness, duplicates và key integrity
def missing_summary(frame: pd.DataFrame) -> pd.DataFrame:
    """Return missing count/rate for one table, sorted by missing rate."""
    result = pd.DataFrame({
        "missing_count": frame.isna().sum(),
        "missing_rate": frame.isna().mean(),
        "dtype": frame.dtypes.astype(str),
    })
    return result.sort_values("missing_rate", ascending=False)

assert train_raw["SK_ID_CURR"].is_unique
assert test_raw["SK_ID_CURR"].is_unique
assert not set(train_raw["SK_ID_CURR"]).intersection(test_raw["SK_ID_CURR"])
print(f"Train duplicate rows: {train_raw.duplicated().sum()}")
print(f"Test duplicate rows: {test_raw.duplicated().sum()}")
display(missing_summary(train_raw).head(20))

In [ ]:
# 5. Application cleaning — không drop missing, chỉ xử lý sentinel đã biết
def clean_application_data(frame: pd.DataFrame) -> pd.DataFrame:
    """Copy application data, flag and null the known DAYS_EMPLOYED sentinel."""
    cleaned = frame.copy()
    if "DAYS_EMPLOYED" in cleaned:
        anomaly = cleaned["DAYS_EMPLOYED"].eq(365243)
        cleaned["DAYS_EMPLOYED_ANOMALOUS"] = anomaly.astype("int8")
        cleaned.loc[anomaly, "DAYS_EMPLOYED"] = np.nan
    cleaned = cleaned.replace([np.inf, -np.inf], np.nan)
    return cleaned

train = clean_application_data(train_raw)
test = clean_application_data(test_raw)
print(f"DAYS_EMPLOYED anomalies: {train['DAYS_EMPLOYED_ANOMALOUS'].sum() if 'DAYS_EMPLOYED_ANOMALOUS' in train else 0}")

In [ ]:
# 6. Notebook-local feature engineering: safe ratios và application groups
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    """Divide aligned series and return NaN for zero/invalid denominators."""
    denominator = denominator.replace(0, np.nan)
    result = numerator.divide(denominator)
    return result.replace([np.inf, -np.inf], np.nan)

def add_application_ratio_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Add transparent amount, affordability and age/employment ratios."""
    result = frame.copy()
    formulas = {
        "CREDIT_INCOME_RATIO": ("AMT_CREDIT", "AMT_INCOME_TOTAL"),
        "ANNUITY_INCOME_RATIO": ("AMT_ANNUITY", "AMT_INCOME_TOTAL"),
        "ANNUITY_CREDIT_RATIO": ("AMT_ANNUITY", "AMT_CREDIT"),
        "GOODS_CREDIT_RATIO": ("AMT_GOODS_PRICE", "AMT_CREDIT"),
        "EMPLOYED_BIRTH_RATIO": ("DAYS_EMPLOYED", "DAYS_BIRTH"),
        "INCOME_PER_PERSON": ("AMT_INCOME_TOTAL", "CNT_FAM_MEMBERS"),
        "CHILDREN_RATIO": ("CNT_CHILDREN", "CNT_FAM_MEMBERS"),
        "CAR_BIRTH_RATIO": ("OWN_CAR_AGE", "DAYS_BIRTH"),
    }
    for name, (numerator, denominator) in formulas.items():
        if numerator in result and denominator in result:
            result[name] = safe_divide(result[numerator], result[denominator])
    return result

def add_ext_source_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Add aggregate and pairwise features from available EXT_SOURCE columns."""
    result = frame.copy()
    columns = [column for column in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"] if column in result]
    if not columns:
        return result
    values = result[columns]
    result["EXT_SOURCE_MEAN"] = values.mean(axis=1)
    result["EXT_SOURCE_MEDIAN"] = values.median(axis=1)
    result["EXT_SOURCE_MIN"] = values.min(axis=1)
    result["EXT_SOURCE_MAX"] = values.max(axis=1)
    result["EXT_SOURCE_STD"] = values.std(axis=1)
    result["EXT_SOURCE_MISSING_COUNT"] = values.isna().sum(axis=1)
    if len(columns) >= 2:
        result["EXT_SOURCE_PRODUCT"] = values.prod(axis=1, min_count=2)
        result["EXT_SOURCE_RANGE"] = result["EXT_SOURCE_MAX"] - result["EXT_SOURCE_MIN"]
    return result

def add_document_contact_housing_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Add counts and selected housing aggregates without target information."""
    result = frame.copy()
    document_columns = [column for column in result if column.startswith("FLAG_DOCUMENT_")]
    contact_columns = [column for column in result if column.startswith("FLAG_CONTACT_")]
    if document_columns:
        result["DOCUMENT_FLAG_COUNT"] = result[document_columns].sum(axis=1)
    if contact_columns:
        result["CONTACT_FLAG_COUNT"] = result[contact_columns].sum(axis=1)
    housing_columns = [column for column in ["APARTMENTS_AVG", "BASEMENTAREA_AVG", "YEARS_BEGINEXPLUATATION_AVG", "ELEVATORS_AVG", "WALLSMATERIAL_MODE"] if column in result]
    numeric_housing = [column for column in housing_columns if pd.api.types.is_numeric_dtype(result[column])]
    if numeric_housing:
        result["HOUSING_AVG_MEAN"] = result[numeric_housing].mean(axis=1)
    return result


In [ ]:
# 7. Build and align application-only feature matrix
def build_application_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Run the ordered application feature steps for one split."""
    result = add_application_ratio_features(frame)
    result = add_ext_source_features(result)
    result = add_document_contact_housing_features(result)
    return result

train_featured = build_application_features(train)
test_featured = build_application_features(test)

target = train_featured.pop("TARGET").astype("int8")
train_ids = train_featured.pop("SK_ID_CURR")
test_ids = test_featured.pop("SK_ID_CURR")

combined = pd.concat([train_featured, test_featured], axis=0, ignore_index=True)
categorical_columns = combined.select_dtypes(include=["object"]).columns.tolist()
for column in categorical_columns:
    combined[column] = combined[column].astype("category")
X_train = combined.iloc[: len(train_featured)].copy()
X_test = combined.iloc[len(train_featured) :].copy()
X_test.index = range(len(X_test))

assert list(X_train.columns) == list(X_test.columns)
assert "TARGET" not in X_train.columns
assert not np.isinf(X_train.select_dtypes(include="number")).any().any()
print(f"Feature count: {X_train.shape[1]}")
print(f"Categorical count: {len(categorical_columns)}")

In [ ]:
# 8. LightGBM baseline: OOF, test predictions, fold metrics, importance
baseline_result = run_lightgbm_cv(
    train_features=X_train,
    target=target,
    test_features=X_test,
    categorical_features=categorical_columns,
    model_config=MODEL_CONFIG,
    validation_config=VALIDATION_CONFIG,
)
print("Fold AUC:", baseline_result["fold_scores"])
print("Mean AUC:", baseline_result["mean_auc"])
print("Std AUC:", baseline_result["std_auc"])
print("OOF AUC:", baseline_result["oof_auc"])
print("Runtime seconds:", round(baseline_result["runtime"], 2))
display(baseline_result["feature_importance"].head(30))

In [ ]:
# 9. Diagnostics và actual artifact export
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(target, baseline_result["oof_predictions"], ax=ax)
ax.set_title(f"OOF ROC curve — AUC {baseline_result['oof_auc']:.5f}")
plt.show()

importance_path = export_dataframe_artifact(
    baseline_result["feature_importance"], OUTPUT_DIR / "feature_importance.csv"
)
metadata_path = export_json_artifact(
    {
        "experiment_name": CONFIG["experiment_name"],
        "feature_count": X_train.shape[1],
        "categorical_features": categorical_columns,
        "fold_scores": baseline_result["fold_scores"],
        "mean_auc": baseline_result["mean_auc"],
        "std_auc": baseline_result["std_auc"],
        "oof_auc": baseline_result["oof_auc"],
        "best_iterations": baseline_result["best_iterations"],
        "runtime": baseline_result["runtime"],
    },
    OUTPUT_DIR / "baseline_metadata.json",
)
print(importance_path)
print(metadata_path)

def installed_version(package_name: str) -> str | None:
    """Return an installed package version without inventing missing values."""
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None

environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "git_commit": GIT_COMMIT,
    "packages": {
        name: installed_version(name)
        for name in ["numpy", "pandas", "scikit-learn", "lightgbm", "matplotlib", "optuna"]
    },
}
fold_metrics = pd.DataFrame(
    {
        "fold": range(1, len(baseline_result["fold_scores"]) + 1),
        "auc": baseline_result["fold_scores"],
        "best_iteration": baseline_result["best_iterations"],
    }
)
oof_frame = pd.DataFrame(
    {
        "SK_ID_CURR": train_ids.to_numpy(),
        "TARGET": target.to_numpy(),
        "OOF_PREDICTION": baseline_result["oof_predictions"],
        "VALIDATION_COUNT": baseline_result["validation_counts"],
    }
)
test_frame = pd.DataFrame(
    {
        "SK_ID_CURR": test_ids.to_numpy(),
        "TEST_PREDICTION": baseline_result["test_predictions"],
    }
)
feature_groups = {
    "application_features": list(X_train.columns),
    "categorical_features": categorical_columns,
}
artifact_paths = {
    "config": export_json_artifact(
        {"config": {**CONFIG, "resolved_data_dir": str(DATA_DIR)}, "mode_config": MODE_CONFIG, "model_config": MODEL_CONFIG, "validation_config": VALIDATION_CONFIG},
        OUTPUT_DIR / "config.json",
    ),
    "environment": export_json_artifact(environment, OUTPUT_DIR / "environment.json"),
    "fold_metrics": export_dataframe_artifact(fold_metrics, OUTPUT_DIR / "fold_metrics.csv"),
    "oof_predictions": export_dataframe_artifact(oof_frame, OUTPUT_DIR / "oof_predictions.csv"),
    "test_predictions": export_dataframe_artifact(test_frame, OUTPUT_DIR / "test_predictions.csv"),
    "feature_importance": importance_path,
}
print({name: str(path) for name, path in artifact_paths.items()})

In [ ]:
# 10. Optional ablation — chỉ chạy khi CONFIG['run_ablation'] = True
FEATURE_GROUPS = {
    "E01_application_clean": [column for column in X_train if not column.startswith(("EXT_SOURCE_", "DOCUMENT_", "CONTACT_", "HOUSING_"))],
    "E02_application_plus_ext_source": list(X_train.columns),
}
ablation_rows = []
if CONFIG["run_ablation"]:
    for experiment_name, columns in FEATURE_GROUPS.items():
        result = run_lightgbm_cv(
            X_train[columns], target, X_test[columns],
            [column for column in categorical_columns if column in columns],
            MODEL_CONFIG, VALIDATION_CONFIG,
        )
        ablation_rows.append({
            "experiment": experiment_name,
            "num_features": len(columns),
            "fold_auc_mean": result["mean_auc"],
            "fold_auc_std": result["std_auc"],
            "oof_auc": result["oof_auc"],
            "runtime": result["runtime"],
        })
ablation = pd.DataFrame(ablation_rows)
display(ablation)

In [ ]:
# 11. Optional tuning — không truyền test data vào objective
if MODE_CONFIG["tuning"]:
    from credit_scoring.modeling.tuning import tune_lightgbm

    SEARCH_SPACE = {
        "learning_rate": {"type": "log_float", "low": 0.005, "high": 0.08},
        "num_leaves": {"type": "int", "low": 16, "high": 96},
        "min_child_samples": {"type": "int", "low": 40, "high": 200},
        "reg_alpha": {"type": "log_float", "low": 1e-3, "high": 2.0},
        "reg_lambda": {"type": "log_float", "low": 0.1, "high": 20.0},
    }
    tuning_result = tune_lightgbm(
        X_train, target, categorical_columns, SEARCH_SPACE,
        {"n_trials": CONFIG["n_trials"], "n_splits": 3, "timeout": 3_600},
    )
    display(tuning_result["trial_dataframe"].sort_values("value", ascending=False).head())
    print(tuning_result["best_params"], tuning_result["best_score"])


In [ ]:
# 12. Submission với schema chính xác của competition
submission_path = None
if MODE_CONFIG["submission"]:
    submission_path = create_home_credit_submission(
        test_ids=test_ids,
        predictions=baseline_result["test_predictions"],
        output_path=OUTPUT_DIR / "submission.csv",
    )
    submission = pd.read_csv(submission_path)
    assert list(submission.columns) == ["SK_ID_CURR", "TARGET"]
    assert len(submission) == len(test_ids)
    print(f"Submission written to: {submission_path}")

if submission_path is not None:
    artifact_paths["submission"] = submission_path
run_metadata_path = OUTPUT_DIR / "run_metadata.json"
artifact_paths["run_metadata"] = run_metadata_path
run_metadata = {
    "experiment_name": CONFIG["experiment_name"],
    "run_mode": CONFIG["run_mode"],
    "git_commit": GIT_COMMIT,
    "model_parameters": MODEL_CONFIG,
    "validation_parameters": VALIDATION_CONFIG,
    "feature_groups": feature_groups,
    "fold_scores": [float(score) for score in baseline_result["fold_scores"]],
    "mean_auc": float(baseline_result["mean_auc"]),
    "std_auc": float(baseline_result["std_auc"]),
    "oof_auc": float(baseline_result["oof_auc"]),
    "runtime": float(baseline_result["runtime"]),
    "artifact_paths": {name: str(path) for name, path in artifact_paths.items()},
}
export_json_artifact(run_metadata, run_metadata_path)
print({name: str(path) for name, path in artifact_paths.items()})

## Kết luận experiment

Chỉ ghi `mean_auc`, `oof_auc`, runtime và artifact vào `docs/experiments/experiment_log.md` sau khi notebook đã chạy hoàn tất. Không ghi metric giả hoặc kết luận dựa trên sample mode.